# 12-phase1-trigger-tuning

Phase 1 (depth-only growing Transformer) 의 **PlateauTrigger ε (epsilon) 튜닝**.

- **배경**: PR #32 (재현성 검증) 에서 발견된 사실 — 현재 ε=0.08 이 학습 σ 보다 커서 trigger 가 **사실상 cooldown 타이머처럼 동작**. 시드 무관 grow_events 정확 일치가 그 증거.
- **검증할 가설**:
  1. **보수화로 plateau 감지 회복**: ε ↓ 시 trigger 가 cooldown 직후 즉시 아닌 진짜 평탄화 시점에 fire
  2. **임계 존재**: ε 너무 작으면 trigger 영원히 fire 안 함 (학습이 계속 변동성 있어 σ ≥ ε)
  3. **최적 범위 존재**: 학습 σ 와 평탄화 σ 사이 어딘가
- **데이터**: TinyShakespeare (char-LM)
- **시드**: 42 (재현성 #2 에서 검증 완료, 단일 시드로 비교 충분)
- **작성일**: 2026-05-25
- **연관**: Issue [#33](https://github.com/EinSofINTEREST/GraphLM/issues/33) / 동기 PR [#32](https://github.com/EinSofINTEREST/GraphLM/pull/32)

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.models.backbone import BackboneConfig, GrowingDecoder
from graphlm.training.loop import TrainConfig, train
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")
    print(f"  total VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 실험 설정

Base (10-phase1) 와 동일 hyperparameters 에서 **`trigger_epsilon` 만 sweep**. 다른 파라미터를 고정해야 epsilon 단독 효과를 볼 수 있음.

Sweep 값:
- `0.08` — Base 값 (cooldown 타이머처럼 동작 → 비교 기준)
- `0.04` — 절반
- `0.02` — 1/4
- `0.01` — 1/8 (너무 보수적일 가능성, fire 0 회 케이스 검증)

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-phase1-trigger-tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
EPS_LIST = [0.08, 0.04, 0.02, 0.01]  # epsilon sweep

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_init_layers=4)
TRAIN_BASE = dict(
    lr=3e-4,
    max_steps=1500,
    max_layers=8,
    trigger_window=100,
    # trigger_epsilon 은 sweep loop 에서 설정
    trigger_cooldown=150,
    trigger_min_history=100,
    device=DEVICE,
)

print("SEED      =", SEED)
print("EPS_LIST  =", EPS_LIST)
for k, v in {**BACKBONE, **TRAIN_BASE}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. epsilon sweep 학습

각 epsilon 값마다 model fresh init + train 1500 step. GPU 에서 1.5분 × 4 ≈ 6분.

In [ ]:
cfg = BackboneConfig(vocab_size=tokenizer.vocab_size, max_seq_len=BLOCK_SIZE, **BACKBONE)

results = {}  # {epsilon: TrainResult}
for eps in EPS_LIST:
    print(f"--- epsilon = {eps} ---")
    set_seed(SEED)
    model = GrowingDecoder(cfg).to(DEVICE)
    data_iter = iter_random_batches(
        dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=SEED
    )
    train_cfg = TrainConfig(**TRAIN_BASE, trigger_epsilon=eps)
    result = train(model, data_iter, train_cfg)
    results[eps] = result
    print(
        f"  done: n_layers={result.final_n_layers}, grow_events={len(result.grow_events)}, "
        f"final_loss≈{result.losses[-1]:.4f}"
    )

## 5. Trigger 거동 분류

각 grow event 의 interval (이전 event 부터의 step 수) 을 cooldown 과 비교:
- **interval ≈ cooldown** → cooldown 직후 즉시 fire = **타이머 동작** (학습 σ < ε 가 상시 만족)
- **interval > cooldown** → 진짜 plateau 감지 = **의도된 동작**
- **fire 0 회** → epsilon 너무 작아 σ < ε 가 절대 만족 X = **과보수**

In [ ]:
COOLDOWN = TRAIN_BASE["trigger_cooldown"]
MIN_HIST = TRAIN_BASE["trigger_min_history"]

print(
    f"{'epsilon':>8}  {'#grow':>6}  {'n_layers':>9}  {'final_loss':>11}  {'intervals':<35}  {'verdict':<22}"
)
print("-" * 110)
for eps, r in results.items():
    # interval 계산 — 첫 event 는 0 부터, 나머지는 직전 event 부터
    prev = 0
    intervals = []
    for step, _n in r.grow_events:
        intervals.append(step - prev)
        prev = step

    # 분류
    # 첫 fire 의 interval 은 min_history 이상 (buffer 채워야 함). 그 외 interval 이 cooldown 과 같으면 timer 동작.
    if not intervals:
        verdict = "NO_FIRE (over-conservative)"
    else:
        post_first = intervals[1:]  # 두 번째 fire 부터 (cooldown 영향)
        n_timer = sum(1 for it in post_first if it == COOLDOWN)
        n_delayed = sum(1 for it in post_first if it > COOLDOWN)
        if not post_first:
            verdict = "single fire only"
        elif n_delayed == 0:
            verdict = f"TIMER ({n_timer} fires @ cooldown)"
        elif n_timer == 0:
            verdict = f"PLATEAU ({n_delayed} fires delayed)"
        else:
            verdict = f"MIXED ({n_timer} timer / {n_delayed} delayed)"

    intervals_str = str(intervals) if intervals else "[]"
    n = min(100, len(r.losses))
    final = sum(r.losses[-n:]) / n if n > 0 else 0.0
    print(
        f"{eps:>8.3f}  {len(r.grow_events):>6}  {r.final_n_layers:>9}  {final:>11.4f}  {intervals_str:<35}  {verdict:<22}"
    )
print()
print(f"cooldown = {COOLDOWN}, min_history = {MIN_HIST}")

## 6. 시각화 — epsilon 별 loss curve + grow event 마커

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(EPS_LIST), 1, figsize=(11, 2.2 * len(EPS_LIST)), sharex=True)
if len(EPS_LIST) == 1:
    axes = [axes]
for ax, (eps, r) in zip(axes, results.items(), strict=True):
    ax.plot(range(1, len(r.losses) + 1), r.losses, lw=0.6, color="steelblue")
    for step, n in r.grow_events:
        ax.axvline(step, color="red", alpha=0.5, lw=0.8)
        ax.text(step, max(r.losses), f" L={n}", color="red", fontsize=7, va="top")
    ax.set_ylabel("CE loss")
    ax.set_title(f"epsilon = {eps}  (#grow={len(r.grow_events)})", fontsize=10)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("step")
fig.tight_layout()
fig.savefig(OUT_DIR / "epsilon_sweep.png", dpi=120)
plt.show()

## 7. 학습 σ vs epsilon 의 직접 비교

Trigger 의 sliding window σ 가 실제 학습 중에 어떤 값을 갖는지 추적 — epsilon 의 적절성 판단 근거.

In [ ]:
import statistics

# baseline (ε=0.08) 의 loss curve 에서 sliding window σ 계산.
# PlateauTrigger 구현체와 일관되게 모분산 (pstdev) 사용 — triggers.py 의
# `var = sum_sq / n - mean**2` 는 분모 n 인 population variance.
baseline = results[EPS_LIST[0]]
window = TRAIN_BASE["trigger_window"]
sigmas = []
for i in range(window, len(baseline.losses)):
    w = baseline.losses[i - window : i]
    sigmas.append(statistics.pstdev(w))

# x 좌표: sigmas[k] = window+k 시점의 σ (직전 window 개 loss 로 계산)
xs = list(range(window, len(baseline.losses)))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(xs, sigmas, lw=0.8, label="sliding window σ (pstdev)")
for eps in EPS_LIST:
    ax.axhline(eps, color="red", alpha=0.4, lw=1.0, linestyle="--", label=f"ε={eps}")
ax.set_xlabel("step")
ax.set_ylabel("sliding window σ (loss)")
ax.set_title(f"실제 학습 σ vs epsilon 후보 (window={window}, pstdev)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "sigma_vs_epsilon.png", dpi=120)
plt.show()

# σ 통계
print(f"학습 σ 통계 (baseline epsilon={EPS_LIST[0]}, pstdev):")
print(f"  σ min  : {min(sigmas):.4f}")
print(f"  σ max  : {max(sigmas):.4f}")
print(f"  σ mean : {statistics.mean(sigmas):.4f}")
print(f"  σ median: {statistics.median(sigmas):.4f}")
print()
print("해석:")
for eps in EPS_LIST:
    pct_below = sum(1 for s in sigmas if s < eps) / len(sigmas) * 100
    print(f"  ε={eps}: 학습 중 σ < ε 가 {pct_below:.1f}% 시간에서 만족")

## 결과 요약 / 권장 epsilon

이 노트북에서 확인할 것:
- §5 의 verdict 컬럼: 각 epsilon 의 trigger 거동 분류 (TIMER / PLATEAU / MIXED / NO_FIRE)
- §7 의 학습 σ 분포: 어느 epsilon 이 학습 σ 와 평탄화 σ 사이에 있는지
- §5 의 final_loss / n_layers: trigger 가 너무 보수적이면 모델이 안 깊어져서 loss 감소 X

**권장 epsilon 결정 기준**:
1. verdict 가 PLATEAU 또는 MIXED (interval > cooldown 인 event 가 다수)
2. final_loss 가 baseline 과 유사하거나 더 낮음
3. §7 에서 σ < ε 가 학습 후반에서만 만족 (초반은 학습 변동성이 커야 정상)

**Phase 1 보완 다음 항목**:
- #4 α 분포 검토 — 학습된 α 가 의도대로 변하는지 (시드별 비교)
- 본 노트북의 권장 epsilon 을 Base scripts (`scripts/train_tinyshakespeare.py`) 의 default 에 반영할지 결정